# 19A4E — Post-test Cycle-25 Score-Shift Diagnostic

## Purpose

Diagnose the strong **2025 operating-threshold collapse** observed after the
primary frozen 2021–2025 evaluation, without changing any scientific decision.

This is a **post-test diagnostic only**.

### Frozen quantities that MUST NOT change
- CNN-GRU weights
- AIA normalisation
- Platt coefficient/intercept
- operating threshold
- primary 2021–2025 scorecard

### Questions
1. Did the raw-score distribution shift in 2025?
2. Did the calibrated-probability distribution shift in 2025?
3. Is ranking still preserved within 2025 despite the operating-point failure?
4. How did positive and negative score distributions move by year?
5. What fraction of positives/negatives lies above the frozen threshold?
6. Is 2025 qualitatively different from 2021–2024 at the score-distribution level?

No new threshold is selected. Any hypothetical threshold shown is forbidden.


In [1]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import ks_2samp
from sklearn.metrics import roc_auc_score, average_precision_score

HOME = Path.home()
PRED = HOME / "aia19_cycle25_final_evaluation" / "cycle25_2021_2025_frozen_predictions.csv.gz"
OUT = HOME / "aia19_cycle25_shift_diagnostic"
OUT.mkdir(parents=True, exist_ok=True)

FROZEN_THRESHOLD = 0.030438695842933242

assert PRED.exists(), PRED
df = pd.read_csv(PRED)

assert len(df) == 49329
assert df["target_sample_id"].nunique() == 49329
assert int(df["y_true"].sum()) == 2351

print("Rows:", len(df))
print("Positives:", int(df["y_true"].sum()))
print("Years:", sorted(df["stored_year"].unique().tolist()))


Rows: 49329
Positives: 2351
Years: [2021, 2022, 2023, 2024, 2025]


## 1. Year-wise score-distribution summary

In [2]:
def q(x, p):
    return float(np.quantile(np.asarray(x, dtype=float), p))

rows = []

for year, g in df.groupby("stored_year"):
    for cls_name, gg in [
        ("all", g),
        ("negative", g[g["y_true"].eq(0)]),
        ("positive", g[g["y_true"].eq(1)]),
    ]:
        rows.append({
            "year": int(year),
            "class": cls_name,
            "n": int(len(gg)),
            "raw_logit_mean": float(gg["raw_logit"].mean()),
            "raw_logit_std": float(gg["raw_logit"].std()),
            "raw_logit_q01": q(gg["raw_logit"], .01),
            "raw_logit_q25": q(gg["raw_logit"], .25),
            "raw_logit_q50": q(gg["raw_logit"], .50),
            "raw_logit_q75": q(gg["raw_logit"], .75),
            "raw_logit_q99": q(gg["raw_logit"], .99),
            "cal_prob_mean": float(gg["calibrated_probability"].mean()),
            "cal_prob_q01": q(gg["calibrated_probability"], .01),
            "cal_prob_q25": q(gg["calibrated_probability"], .25),
            "cal_prob_q50": q(gg["calibrated_probability"], .50),
            "cal_prob_q75": q(gg["calibrated_probability"], .75),
            "cal_prob_q99": q(gg["calibrated_probability"], .99),
            "fraction_above_frozen_threshold": float(
                (gg["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
            ),
        })

summary = pd.DataFrame(rows)
summary.to_csv(OUT / "year_class_score_distribution_summary.csv", index=False)

print(summary.to_string(index=False))


 year    class     n  raw_logit_mean  raw_logit_std  raw_logit_q01  raw_logit_q25  raw_logit_q50  raw_logit_q75  raw_logit_q99  cal_prob_mean  cal_prob_q01  cal_prob_q25  cal_prob_q50  cal_prob_q75  cal_prob_q99  fraction_above_frozen_threshold
 2021      all  5104       -1.014349       1.268819      -2.737898      -2.322956      -1.021190       0.208812       0.993212       0.017785      0.005099      0.006490      0.013790      0.027902      0.043433                         0.210031
 2021 negative  4978       -1.027832       1.270416      -2.738094      -2.337388      -1.052874       0.197865       0.993465       0.017671      0.005098      0.006436      0.013541      0.027729      0.043439                         0.207513
 2021 positive   126       -0.481683       1.080771      -2.449943      -1.320856      -0.486936       0.471840       0.982765       0.022316      0.006029      0.011605      0.018752      0.032390      0.043180                         0.309524
 2022      all  9999

## 2. Operating-point decomposition by year

In [3]:
op_rows = []

for year, g in df.groupby("stored_year"):
    pos = g[g["y_true"].eq(1)]
    neg = g[g["y_true"].eq(0)]

    op_rows.append({
        "year": int(year),
        "n": int(len(g)),
        "positives": int(len(pos)),
        "prevalence": float(g["y_true"].mean()),
        "positive_fraction_above_threshold": float(
            (pos["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
        ),
        "negative_fraction_above_threshold": float(
            (neg["calibrated_probability"] >= FROZEN_THRESHOLD).mean()
        ),
        "positive_median_calibrated_probability": float(
            pos["calibrated_probability"].median()
        ),
        "negative_median_calibrated_probability": float(
            neg["calibrated_probability"].median()
        ),
        "positive_median_raw_logit": float(pos["raw_logit"].median()),
        "negative_median_raw_logit": float(neg["raw_logit"].median()),
        "roc_auc": float(roc_auc_score(g["y_true"], g["calibrated_probability"])),
        "pr_auc": float(average_precision_score(g["y_true"], g["calibrated_probability"])),
    })

op = pd.DataFrame(op_rows)
op.to_csv(OUT / "yearwise_operating_point_decomposition.csv", index=False)

print(op.to_string(index=False))


 year     n  positives  prevalence  positive_fraction_above_threshold  negative_fraction_above_threshold  positive_median_calibrated_probability  negative_median_calibrated_probability  positive_median_raw_logit  negative_median_raw_logit  roc_auc   pr_auc
 2021  5104        126    0.024687                           0.309524                           0.207513                                0.018752                                0.013541                  -0.486936                  -1.052874 0.632295 0.038003
 2022  9999        119    0.011901                           0.394958                           0.344231                                0.025225                                0.018661                   0.031584                  -0.495422 0.597992 0.016635
 2023 12392        278    0.022434                           0.643885                           0.397969                                0.035653                                0.024246                   0.641763                  

## 3. Distribution-shift tests against 2024

These tests are descriptive diagnostics only.

We compare 2025 with 2024 using two-sample Kolmogorov–Smirnov statistics for:
- raw logits;
- calibrated probabilities;
- positives only;
- negatives only.

A small p-value indicates that the score distributions differ, but does not by itself identify the physical cause.


In [4]:
g24 = df[df["stored_year"].eq(2024)]
g25 = df[df["stored_year"].eq(2025)]

tests = []

for col in ["raw_logit", "calibrated_probability"]:
    for cls_name, a, b in [
        ("all", g24, g25),
        ("positive", g24[g24["y_true"].eq(1)], g25[g25["y_true"].eq(1)]),
        ("negative", g24[g24["y_true"].eq(0)], g25[g25["y_true"].eq(0)]),
    ]:
        stat, pvalue = ks_2samp(a[col].to_numpy(), b[col].to_numpy())
        tests.append({
            "reference_year": 2024,
            "comparison_year": 2025,
            "class": cls_name,
            "variable": col,
            "ks_statistic": float(stat),
            "p_value": float(pvalue),
            "n_2024": int(len(a)),
            "n_2025": int(len(b)),
        })

ks = pd.DataFrame(tests)
ks.to_csv(OUT / "ks_2024_vs_2025_score_shift.csv", index=False)
print(ks.to_string(index=False))


 reference_year  comparison_year    class               variable  ks_statistic       p_value  n_2024  n_2025
           2024             2025      all              raw_logit      0.645481  0.000000e+00   12553    9281
           2024             2025 positive              raw_logit      0.768807 2.002477e-217    1316     512
           2024             2025 negative              raw_logit      0.652294  0.000000e+00   11237    8769
           2024             2025      all calibrated_probability      0.645481  0.000000e+00   12553    9281
           2024             2025 positive calibrated_probability      0.768807 2.002477e-217    1316     512
           2024             2025 negative calibrated_probability      0.652294  0.000000e+00   11237    8769


## 4. Score distributions by year

In [5]:
fig, ax = plt.subplots(figsize=(9, 5))

for year in sorted(df["stored_year"].unique()):
    vals = df.loc[df["stored_year"].eq(year), "calibrated_probability"].to_numpy()
    ax.hist(
        vals,
        bins=120,
        histtype="step",
        density=True,
        label=str(year),
    )

ax.axvline(FROZEN_THRESHOLD, linestyle="--", linewidth=1.5, label="Frozen threshold")
ax.set_xlim(0, np.quantile(df["calibrated_probability"], 0.995))
ax.set_xlabel("Calibrated probability")
ax.set_ylabel("Density")
ax.set_title("Frozen Cycle-25 calibrated-probability distributions")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "calibrated_probability_distribution_by_year.png", dpi=180)
plt.close(fig)

print("Saved calibrated_probability_distribution_by_year.png")


Saved calibrated_probability_distribution_by_year.png


## 5. Positive-class distributions by year

In [6]:
fig, ax = plt.subplots(figsize=(9, 5))

pos_df = df[df["y_true"].eq(1)]

for year in sorted(pos_df["stored_year"].unique()):
    vals = pos_df.loc[pos_df["stored_year"].eq(year), "calibrated_probability"].to_numpy()
    ax.hist(
        vals,
        bins=100,
        histtype="step",
        density=True,
        label=str(year),
    )

ax.axvline(FROZEN_THRESHOLD, linestyle="--", linewidth=1.5, label="Frozen threshold")
ax.set_xlim(0, np.quantile(pos_df["calibrated_probability"], 0.995))
ax.set_xlabel("Calibrated probability")
ax.set_ylabel("Density")
ax.set_title("Positive-class calibrated-probability distributions")
ax.legend()
fig.tight_layout()
fig.savefig(OUT / "positive_calibrated_probability_distribution_by_year.png", dpi=180)
plt.close(fig)

print("Saved positive_calibrated_probability_distribution_by_year.png")


Saved positive_calibrated_probability_distribution_by_year.png


## 6. Yearly probability quantiles

In [7]:
quantile_rows = []

for year, g in df.groupby("stored_year"):
    for label in [0, 1]:
        gg = g[g["y_true"].eq(label)]
        rec = {
            "year": int(year),
            "y_true": int(label),
            "n": int(len(gg)),
        }
        for p in [0.01,0.05,0.10,0.25,0.50,0.75,0.90,0.95,0.99]:
            rec[f"q{int(p*100):02d}"] = float(
                np.quantile(gg["calibrated_probability"], p)
            )
        quantile_rows.append(rec)

quantiles = pd.DataFrame(quantile_rows)
quantiles.to_csv(OUT / "year_class_calibrated_probability_quantiles.csv", index=False)
print(quantiles.to_string(index=False))


 year  y_true     n      q01      q05      q10      q25      q50      q75      q90      q95      q99
 2021       0  4978 0.005098 0.005234 0.005397 0.006436 0.013541 0.027729 0.037052 0.040400 0.043439
 2021       1   126 0.006029 0.006605 0.007290 0.011605 0.018752 0.032390 0.041584 0.042314 0.043180
 2022       0  9880 0.005166 0.005390 0.005714 0.007870 0.018661 0.036360 0.041727 0.042977 0.043977
 2022       1   119 0.005831 0.006311 0.009110 0.016318 0.025225 0.037563 0.041896 0.043934 0.043980
 2023       0 12114 0.005214 0.005611 0.006138 0.010973 0.024246 0.037526 0.041694 0.042800 0.043795
 2023       1   278 0.005752 0.011581 0.016511 0.024878 0.035653 0.041137 0.042408 0.043106 0.043619
 2024       0 11237 0.005180 0.005450 0.005730 0.008293 0.018869 0.035987 0.041693 0.042895 0.043814
 2024       1  1316 0.006030 0.009391 0.012390 0.026875 0.038478 0.042058 0.043489 0.043940 0.044140
 2025       0  8769 0.005193 0.005299 0.005371 0.005518 0.005730 0.006077 0.012048 0.021814

## 7. Diagnostic interpretation record

In [8]:
op_idx = op.set_index("year")

diagnostic = {
    "status": "POST_TEST_SCORE_SHIFT_DIAGNOSTIC_COMPLETE_NO_TUNING",
    "primary_test_result_unchanged": True,
    "frozen_threshold": FROZEN_THRESHOLD,
    "year_2024": op_idx.loc[2024].to_dict(),
    "year_2025": op_idx.loc[2025].to_dict(),
    "ks_2024_vs_2025": ks.to_dict(orient="records"),
    "model_weights_updated": False,
    "calibrator_refit": False,
    "threshold_reselected": False,
    "metrics_reoptimised": False,
    "notes": [
        "This notebook is diagnostic only and does not alter the frozen 19A4 result.",
        "ROC-AUC/PR-AUC characterize ranking; the frozen operating point characterizes threshold transfer.",
        "A large shift in positive score distribution with preserved ranking is consistent with score/probability distribution shift.",
        "Statistical distribution shift alone does not establish whether the cause is solar-regime shift, acquisition/preprocessing shift, or both.",
        "AIA source-image statistics and production metadata should be compared separately before attributing cause."
    ],
}

(OUT / "diagnostic_record.json").write_text(json.dumps(diagnostic, indent=2) + "\n")

print(json.dumps(diagnostic, indent=2))
print("\n19A4E_SCORE_SHIFT_DIAGNOSTIC_COMPLETE")


{
  "status": "POST_TEST_SCORE_SHIFT_DIAGNOSTIC_COMPLETE_NO_TUNING",
  "primary_test_result_unchanged": true,
  "frozen_threshold": 0.030438695842933242,
  "year_2024": {
    "n": 12553.0,
    "positives": 1316.0,
    "prevalence": 0.10483549749063968,
    "positive_fraction_above_threshold": 0.7013677811550152,
    "negative_fraction_above_threshold": 0.34279611996084364,
    "positive_median_calibrated_probability": 0.03847834692238675,
    "negative_median_calibrated_probability": 0.018868837787244,
    "positive_median_raw_logit": 0.7772150635719299,
    "negative_median_raw_logit": -0.476082295179367,
    "roc_auc": 0.7372442941833767,
    "pr_auc": 0.23444630426879465
  },
  "year_2025": {
    "n": 9281.0,
    "positives": 512.0,
    "prevalence": 0.05516646913048163,
    "positive_fraction_above_threshold": 0.0078125,
    "negative_fraction_above_threshold": 0.02942182689018132,
    "positive_median_calibrated_probability": 0.00716251656219755,
    "negative_median_calibrated_pr